# Experiment 1 — the shuffle behind `groupBy`

Spark UI: http://localhost:4040 → **Stages** (Summary Metrics, Tasks table) and **SQL / DataFrame**.

In [7]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m3-exp1-groupby").getOrCreate()
big = spark.read.parquet("/opt/data/m3/orders_big")
print("input partitions:", big.rdd.getNumPartitions())

def timed(label, df, path):
    t0 = time.perf_counter()
    df.write.mode("overwrite").parquet(path)
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

[Stage 0:>                                                          (0 + 1) / 1]

input partitions: 16


## A — plain groupBy

**PREDICT:** how many `Exchange` nodes and where; how many `HashAggregate` nodes and on which side of the Exchange; tasks in each stage; roughly how much shuffle write compared with the input size.

In [8]:
agg_a = big.groupBy("customer_id").agg(F.count("*").alias("n"), F.sum("amount").alias("total"))
agg_a.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [customer_id#38, amount#40]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
ReadSchema: struct<customer_id:int,amount:double>

(2) HashAggregate
Input [2]: [customer_id#38, amount#40]
Keys [1]: [customer_id#38]
Functions [2]: [partial_count(1), partial_sum(amount#40)]
Aggregate Attributes [2]: [count#54L, sum#55]
Results [3]: [customer_id#38, count#56L, sum#57]

(3) Exchange
Input [3]: [customer_id#38, count#56L, sum#57]
Arguments: hashpartitioning(customer_id#38, 200), ENSURE_REQUIREMENTS, [plan_id=185]

(4) HashAggregate
Input [3]: [customer_id#38, count#56L, sum#57]
Keys [1]: [customer_id#38]
Functions [2]: [count(1), sum(amount#40)]
Aggregate Attributes [2]: [count(1)#51L, sum(amount#40)#52]
Results [3]: [customer_id#38, count(1)#51L AS n#43L, sum(amount#40)#52 AS total#44]

(5) AdaptiveSp

In [12]:
timed("A plain groupBy", agg_a, "/opt/data/m3/out/agg_a")

A plain groupBy: 0.6s


## B — repartition by the key first, then the same groupBy

**PREDICT:** how many `Exchange` nodes now; how many `HashAggregate` nodes; shuffle write compared with A; runtime compared with A.

In [10]:
agg_b = big.repartition("customer_id").groupBy("customer_id").agg(F.count("*").alias("n"), F.sum("amount").alias("total"))
agg_b.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- HashAggregate (3)
      +- Exchange (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [customer_id#38, amount#40]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
ReadSchema: struct<customer_id:int,amount:double>

(2) Exchange
Input [2]: [customer_id#38, amount#40]
Arguments: hashpartitioning(customer_id#38, 200), REPARTITION_BY_COL, [plan_id=263]

(3) HashAggregate
Input [2]: [customer_id#38, amount#40]
Keys [1]: [customer_id#38]
Functions [2]: [partial_count(1), partial_sum(amount#40)]
Aggregate Attributes [2]: [count#70L, sum#71]
Results [3]: [customer_id#38, count#72L, sum#73]

(4) HashAggregate
Input [3]: [customer_id#38, count#72L, sum#73]
Keys [1]: [customer_id#38]
Functions [2]: [count(1), sum(amount#40)]
Aggregate Attributes [2]: [count(1)#68L, sum(amount#40)#69]
Results [3]: [customer_id#38, count(1)#68L AS n#60L, sum(amount#40)#69 AS total#61]

(5) AdaptiveSparkPlan
O

In [13]:
timed("B repartition then groupBy", agg_b, "/opt/data/m3/out/agg_b")

[Stage 10:================================>                        (9 + 7) / 16]

B repartition then groupBy: 1.0s


## Observe

Stages tab, for each run: the map stage's *Shuffle Write* (total and per task), the reduce stage's *Shuffle Read*, task counts, durations.

## Stop before opening the next notebook

In [6]:
spark.stop()